In [ ]:
import os
from pathlib import Path
import random
import shutil
import json

os.environ["KAGGLE_USERNAME"] = "your_name"
os.environ["KAGGLE_KEY"] = "your_api_key"

!pip install -q kaggle scikit-learn seaborn optuna optuna-integration[tfkeras]

if not Path("/content/Training").exists() or not Path("/content/Testing").exists():
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content --unzip

In [ ]:
random.seed(42)

src_train = Path("/content/Training")
src_test  = Path("/content/Testing")
dst_root  = Path("/content/data")
dst_train, dst_val, dst_test = dst_root/"train", dst_root/"val", dst_root/"test"

if dst_root.exists():
    shutil.rmtree(dst_root)
shutil.copytree(src_test, dst_test)

VAL_FRAC = 0.15
classes  = sorted([d.name for d in src_train.iterdir() if d.is_dir()])

for c in classes:
    imgs = [p for p in (src_train / c).glob("*") if p.is_file()]
    random.shuffle(imgs)
    n_val = int(len(imgs) * VAL_FRAC)
    (dst_train / c).mkdir(parents=True, exist_ok=True)
    (dst_val   / c).mkdir(parents=True, exist_ok=True)
    for p in imgs[:n_val]:
        shutil.copy2(p, dst_val / c / p.name)
    for p in imgs[n_val:]:
        shutil.copy2(p, dst_train / c / p.name)

for split in ["train", "val", "test"]:
    base = dst_root / split
    print(split, {c.name: len(list(c.glob("*"))) for c in base.iterdir()})

In [ ]:
import optuna
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report, f1_score

tf.keras.utils.set_random_seed(42)

In [ ]:
IMG_SIZE  = (256, 256)
SEED      = 42
AUTOTUNE  = tf.data.AUTOTUNE

train_dir, val_dir, test_dir = str(dst_train), str(dst_val), str(dst_test)

N_TRIALS       = 8
PRUNING_EPOCHS = 5
FINAL_EPOCHS   = 30

# DenseNet121 has 427 layers; search covers ~2–37% of the network
UNFREEZE_MIN = 10
UNFREEZE_MAX = 160

In [ ]:
def load_datasets(batch_size: int):
    kw = dict(label_mode="categorical", image_size=IMG_SIZE, batch_size=batch_size)
    train_ds = tf.keras.utils.image_dataset_from_directory(train_dir, shuffle=True, seed=SEED, **kw)
    val_ds   = tf.keras.utils.image_dataset_from_directory(val_dir,   shuffle=False, **kw)
    test_ds  = tf.keras.utils.image_dataset_from_directory(test_dir,  shuffle=False, **kw)
    class_names = train_ds.class_names
    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds   = val_ds.cache().prefetch(AUTOTUNE)
    test_ds  = test_ds.cache().prefetch(AUTOTUNE)
    return train_ds, val_ds, test_ds, class_names

In [ ]:
def build_model(num_classes, learning_rate, unfreeze_last_n, dropout_rate, focal_gamma, label_smoothing):
    augment = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.2),
    ], name="augmentation")

    base = tf.keras.applications.DenseNet121(include_top=False, weights="imagenet", input_shape=IMG_SIZE+(3,))
    base.trainable = True
    for layer in base.layers[:-unfreeze_last_n]:
        layer.trainable = False

    inputs  = layers.Input(shape=IMG_SIZE+(3,))
    x       = augment(inputs)
    x       = tf.keras.applications.densenet.preprocess_input(x)
    x       = base(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=focal_gamma, label_smoothing=label_smoothing),
        metrics=["accuracy", tf.keras.metrics.F1Score(average="macro", name="f1_macro")],
    )
    return model

In [ ]:
def objective(trial: optuna.Trial) -> float:
    lr      = trial.suggest_float("learning_rate",   1e-6, 1e-3, log=True)
    unfreeze= trial.suggest_int(  "unfreeze_last_n", UNFREEZE_MIN, UNFREEZE_MAX, step=10)
    dropout = trial.suggest_float("dropout_rate",    0.1,  0.6)
    gamma   = trial.suggest_float("focal_gamma",     0.5,  4.0)
    smooth  = trial.suggest_float("label_smoothing", 0.0,  0.15)
    bs      = trial.suggest_categorical("batch_size", [16, 32])

    train_ds, val_ds, _, class_names = load_datasets(bs)
    model = build_model(len(class_names), lr, unfreeze, dropout, gamma, smooth)

    history = model.fit(
        train_ds, validation_data=val_ds, epochs=PRUNING_EPOCHS, verbose=0,
        callbacks=[
            optuna.integration.TFKerasPruningCallback(trial, monitor="val_f1_macro"),
            tf.keras.callbacks.EarlyStopping(monitor="val_f1_macro", patience=3, mode="max", restore_best_weights=True),
        ],
    )
    return max(history.history.get("val_f1_macro", [0.0]))

In [ ]:
# seed=SEED+1 ensures DenseNet explores different initial configs than ResNet50
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED + 1),
    pruner=optuna.pruners.HyperbandPruner(min_resource=2, max_resource=PRUNING_EPOCHS, reduction_factor=3),
    study_name="densenet121_brain_tumor",
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
best_val_f1 = study.best_value
print(best_params, f"val_f1={best_val_f1:.4f}")

try:
    optuna.visualization.plot_optimization_history(study).show()
    optuna.visualization.plot_param_importances(study).show()
except Exception:
    pass

In [ ]:
train_ds, val_ds, test_ds, class_names = load_datasets(best_params["batch_size"])
num_classes = len(class_names)

final_model = build_model(num_classes, **{k: best_params[k] for k in
    ["learning_rate","unfreeze_last_n","dropout_rate","focal_gamma","label_smoothing"]})
final_model.summary()

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("densenet121_best.keras", monitor="val_f1_macro", save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_f1_macro", patience=5, restore_best_weights=True, mode="max", verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
]
history = final_model.fit(train_ds, validation_data=val_ds, epochs=FINAL_EPOCHS, callbacks=callbacks)

In [ ]:
best = tf.keras.models.load_model("densenet121_best.keras")
test_loss, test_acc, test_f1 = best.evaluate(test_ds, verbose=1)
print(f"loss={test_loss:.4f}  acc={test_acc:.4f}  f1={test_f1:.4f}")

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
y_pred = np.argmax(best.predict(test_ds, verbose=1), axis=1)

cm = confusion_matrix(y_true, y_pred)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print(f"Macro F1 (sklearn): {f1_score(y_true, y_pred, average='macro'):.4f}")

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix – DenseNet121")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (tr, val, title) in zip(axes, [
    ("loss",     "val_loss",     "Loss"),
    ("accuracy", "val_accuracy", "Accuracy"),
    ("f1_macro", "val_f1_macro", "Macro F1"),
]):
    ax.plot(history.history[tr],  label="Train")
    ax.plot(history.history[val], label="Val")
    ax.legend(); ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
config = {
    "model": "DenseNet121", "image_size": list(IMG_SIZE), "epochs": FINAL_EPOCHS,
    "classes": class_names, "preprocessing": "densenet.preprocess_input",
    "hpo_method": "Optuna TPE + HyperbandPruner", "n_trials": N_TRIALS,
    "best_val_f1_hpo": best_val_f1, "unfreeze_range": f"{UNFREEZE_MIN}-{UNFREEZE_MAX}",
    "sampler_seed": SEED + 1, "test_loss": float(test_loss),
    "test_accuracy": float(test_acc), "test_f1_macro": float(test_f1),
    **best_params,
}
with open("densenet121_config.json",    "w") as f: json.dump(config,      f, indent=2)
with open("densenet121_class_names.json","w") as f: json.dump(class_names, f, indent=2)
best.save("densenet121_best.keras")
print(os.listdir())

In [ ]:
from google.colab import files
files.download("densenet121_best.keras")